# Patron de Reflexion: un agente que se autocritica

Este notebook muestra el patron de **reflexion** (generar -> criticar -> revisar): un LLM produce una respuesta, otro rol del mismo modelo (el "revisor") la critica, y si encuentra un problema el modelo la corrige. Se repite hasta que el revisor la aprueba o se agotan los intentos.

Caso de uso: un agente que explica conceptos de fisica a un estudiante, y se autocorrige antes de entregar la respuesta final.

Las dos "voces" (explicador y revisor) son el mismo LLM con distintas instrucciones de sistema.

Requisitos: Ollama corriendo localmente con al menos el modelo `llama3.2:1b` descargado (ver `ayuda.txt`).

In [1]:
from reflection_agent import explicar, revisar, corregir, run_reflection
from display_helpers import show_trace

## 1. Sin reflexion (linea base)

Primero pedimos la explicacion una sola vez, sin ningun paso de revision. Con un modelo pequeno como `llama3.2:1b`, a veces la explicacion tiene errores o imprecisiones que pasan sin que nadie las revise.

In [2]:
concepto = "por que el periodo de un pendulo simple no depende de la masa"
model = "llama3.2:1b"

explicacion_base = explicar(concepto, model)
print(explicacion_base)

¡Claro! El periodo de un pendulo simple es una medida que describe la duración de un movimiento circular en un pendulo simple. La razón por la cual el periodo no depende de la masa es que el pendulo simple se mueve en un círculo sin perder energía. La masa de la pendulo no juega un papel importante en este movimiento, ya que la energía cinética y potencial son las que determinan la velocidad y la aceleración de la pendulo.

En otras palabras, la pendulo se mueve en un círculo porque la fuerza de la gravedad actúa sobre ella, lo que la impulsa a cambiar su posición. La masa de la pendulo no cambia, pero la fuerza de la gravedad sí cambia con la altura de la pendulo. Esto significa que, aunque la masa sea mayor en un pendulo con un ancho más grande, la pendulo puede moverse en un círculo con el mismo periodo que un pendulo con un ancho menor, siempre y cuando la fuerza de la gravedad sea la misma.


## 2. El ciclo paso a paso

El patron tiene tres funciones en `reflection_agent.py`:

- `explicar(concepto, model)` -- genera una primera explicacion.
- `revisar(concepto, explicacion, model)` -- el mismo modelo, con otra instruccion de sistema, actua como revisor: dice si la aprueba y por que no si la rechaza.
- `corregir(concepto, explicacion, comentarios, model)` -- vuelve a generar la explicacion, esta vez viendo los comentarios del revisor.

Vamos a correrlas a mano una vez, sobre la explicacion de la seccion anterior.

In [3]:
revision = revisar(concepto, explicacion_base, model)
print("Aprobada:", revision["aprobada"])
print(revision["comentarios"])

Aprobada: False
APROBADA

COMENTARIOS: El profesor ha proporcionado una explicación clara y concisa sobre la razón por la cual el periodo del pendulo simple no depende de la masa. La descripción del movimiento circular y la importancia de la fuerza de la gravedad en el pendulo es correcta. Sin embargo, podría ser más claro el argumento de que la masa no juega un papel importante en este movimiento, ya que la pendulo se mueve en un círculo porque la fuerza de la gravedad actúa sobre ella, lo que la impulsa a cambiar su posición.


In [4]:
if not revision["aprobada"]:
    explicacion_corregida = corregir(concepto, explicacion_base, revision["comentarios"], model)
    print(explicacion_corregida)
else:
    print("El revisor la aprobo, no hace falta corregir.")

Tienes absolutamente razón, la ecuación del periodo T no depende directamente de la masa de la tierra. La constancia de la masa del pendulo (m) se refleja en la ecuación cuando sustituimos la constante de gravitación universal (G) y la masa de la tierra (M) en la ecuación.

La ecuación se simplifica a:

T = 2π√(G × M / d)

donde:

* T es el periodo del pendulo simple.
* G es la constante de gravitación universal (1,62 × 10^-8 m^3 kg^-1 s^-2).
* M es la masa de la tierra (1,99 × 10^30 kg).
* d es la distancia entre el centro del pendulo y el punto de acción de la fuerza de gravedad (la tierra).

Por lo tanto, la ecuación del periodo T no depende directamente de la masa de la tierra. La masa de la tierra (M) es una constante que se refleja en la ecuación, y la constancia de la masa del pendulo (m) no afecta directamente el resultado.


## 3. El ciclo completo: `run_reflection`

`run_reflection` encadena estos tres pasos automaticamente hasta `max_rounds` veces, o hasta que el revisor aprueba. Devuelve la traza completa (cada ronda con su explicacion y su revision) para poder inspeccionar que paso.

In [4]:
trace = run_reflection(concepto, model=model, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

El periodo de un pendulo simple es una medida de su velocidad relativa y no está relacionado con la masa. La razón es que el periodo es una función del momento de inertia del pendulo y de su aceleración. La masa no juega un papel directo en la determinación del periodo.

Imagina un pendulo simple con una masa en el centro. La energia cinética del pendulo se distribuye en forma de kinetic energy (energia de movimiento) y potencial. La energía potencial se crea en el centro del pendulo debido a su inclinación y la energía cinética se absorbe del pendulo a medida que se mueve hacia adelante.

Si la masa aumenta significativamente, como en el caso de un pendulo con una masa mucho más grande, el centro del pendulo se mueve más rápidamente debido a la mayor atraencia de la masa. Esto significa que la energía potencial que creaba en el centro del pendulo se absorbe más rápidamente, lo que significa que el pendulo necesita moverse más rápido para mantener la misma energía potencial.

En otras palabras, la masa no afecta la velocidad relativa del pendulo, sino que afecta la energía potencial y la energía cinética. Por lo tanto, la masa no influye en la determinación del periodo de un pendulo simple.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA
La explicación del profesor es correcta y bien explicada, sin embargo, puedo sugerir un poco de claridad adicional. La frase "la energía potencial se absorbe más rápidamente" podría ser reemplazada por "la energía potencial se absorbe más rápidamente debido a la mayor atraencia de la masa" para reflejar la complejidad del problema.

### Ronda 1

La explicación del profesor es correcta y bien explicada, sin embargo, puedo sugerir un poco de claridad adicional. La frase "la energía potencial se absorbe más rápidamente" podría ser reemplazada por "la energía potencial se absorbe más rápidamente debido a la mayor atraencia de la masa" para reflejar la complejidad del problema.

Además, una forma más clara de expresar la idea sería utilizar el concepto de "aceleración" y "inercia angular" en lugar de "energía potencial" y "energía cinética". La masa juega un papel importante en la determinación del periodo del pendulo simple, ya que afecta la inercia angular y la aceleración del pendulo.

La explicación de forma completa sería:

La explicación del profesor es correcta y bien explicada, sin embargo, puedo sugerir un poco de claridad adicional. La frase "la energía potencial se absorbe más rápidamente" podría ser reemplazada por "la energía potencial se absorbe más rápidamente debido a la mayor atraencia de la masa" para reflejar la complejidad del problema.

La masa no afecta la velocidad relativa del pendulo, sino que afecta la inercia angular y la aceleración. La inercia angular es la capacidad del pendulo para mantenerse en su estado original sin movimiento, mientras que la aceleración es la relación entre la fuerza aplicada y la masa del pendulo. La masa juega un papel importante en la determinación del periodo del pendulo simple, ya que afecta la aceleración del pendulo y, por lo tanto, su inercia angular.

En otras palabras, la masa no influye en la determinación del periodo de un pendulo simple. La masa no afecta la velocidad relativa del pendulo, sino que afecta la energía potencial y la energía cinética. Por lo tanto, la masa no influye en la determinación del periodo de un pendulo simple.

... (2 outputs left out)

La explicación original es correcta, pero sí hay algunas sugerencias para mejorarla:

La explicación original es correcta, sin embargo, puedo sugerir un poco de claridad adicional. La frase "la energía potencial se absorbe más rápidamente" podría ser reemplazada por "la energía potencial se absorbe más rápidamente debido a la mayor atraencia de la masa" para reflejar la complejidad del problema.

La masa no afecta la velocidad relativa del pendulo, sino que afecta la inercia angular y la aceleración. La inercia angular es la capacidad del pendulo para mantenerse en su estado original sin movimiento, mientras que la aceleración es la relación entre la fuerza aplicada y la masa del pendulo. La masa juega un papel importante en la determinación del periodo del pendulo simple, ya que afecta la aceleración del pendulo y, por lo tanto, su inercia angular.

Además, la masa no afecta la energía potencial y la energía cinética de los pendulios. La energía potencial es la cantidad de energía que tiene un objeto en un momento determinado, y se manifiesta en la forma de la energía cinética, que es la energía que tiene un objeto en movimiento. La masa de un objeto no afecta la cantidad de energía potencial o la energía cinética, ya que estos dos conceptos están relacionados con la posición y la velocidad relativas del objeto, no con su masa.

En otras palabras, la masa no influye en la determinación del periodo de un pendulo simple. La masa no afecta la velocidad relativa del pendulo, sino que afecta la aceleración del pendulo y, por lo tanto, su inercia angular.

La explicación de forma completa sería:

La masa no afecta la velocidad relativa del pendulo, sino que afecta la inercia angular y la aceleración. La inercia angular es la capacidad del pendulo para mantenerse en su estado original sin movimiento, mientras que la aceleración es la relación entre la fuerza aplicada y la masa del pendulo.

La masa juega un papel importante en la determinación del periodo del pendulo simple, ya que afecta la aceleración del pendulo y, por lo tanto, su inercia angular.

Además, la masa no afecta la energía potencial y la energía cinética de los pendulios. La energía potencial es la cantidad de energía que tiene un objeto en un momento determinado, y se manifiesta en la forma de la energía cinética, que es la energía que tiene un objeto en movimiento.

Por lo tanto, la masa no influye en la determinación del periodo de un pendulo simple, y la masa solo afecta la inercia angular y la aceleración.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA: 
COMENTARIOS: La sugerencia de reemplazar la frase "la energía potencial se absorbe más rápidamente" por "la energía potencial se absorbe más rápidamente debido a la mayor atraencia de la masa" es una mejora significativa de la explicación, ya que refleja con más precisión la complejidad del problema. Sin embargo, la respuesta final debería ser de un solo párrafo.

### Ronda 3

La masa no afecta la velocidad relativa del pendulo, sino que afecta la inercia angular y la aceleración. La inercia angular es la capacidad del pendulo para mantenerse en su estado original sin movimiento, mientras que la aceleración es la relación entre la fuerza aplicada y la masa del pendulo. La masa juega un papel importante en la determinación del periodo del pendulo simple, ya que afecta la aceleración del pendulo y, por lo tanto, su inercia angular.

## 4. Varios conceptos

Probemos con un par de conceptos mas -- el codigo no cambia, solo el texto que le pasamos.

In [6]:
conceptos = [
    "por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo",
    "diferencia entre velocidad y aceleracion",
]

for c in conceptos:
    trace = run_reflection(c, model=model, max_rounds=3)
    show_trace(c, trace)

## Concepto: por que un objeto en caida libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo

### Ronda 0

Excelente pregunta, estudiante. Vamos a analizar el caso de cómo un objeto cae desde una altura y un objeto lanzado horizontalmente desde la misma altura se tocan.

Quiero explicarte un concepto llamado "aceleración" (en inglés, acceleration). Es una fuerza que actúa en cualquier objeto que se mueve de una velocidad inicial a una velocidad más alta. En el caso de los dos objetos, la aceleración es la misma en ambos casos.

Imagina que el objeto caído desde la altura se mueve en una línea recta, con una velocidad inicial. A medida que cae, la aceleración de su proyectil es la gravedad, que es la fuerza que actúa a través de la distancia (en este caso, la altura). La aceleración es constante, es decir, no cambia con el tiempo, y es igual a la razón entre la altura de la caída y el tiempo que tarda en caer.

Por otro lado, el objeto lanzado horizontalmente desde la misma altura también se mueve en una línea recta, pero su velocidad aumenta constantemente debido a la fuerza del impulso. La aceleración en este caso es la velocidad inicial (la velocidad al lanzar el objeto) dividida por el tiempo que tarda en caer (la altura).

Ahora, la clave está en la velocidad inicial. El objeto caído desde la altura tiene una velocidad inicial negativa, es decir, se mueve hacia atrás. Por otro lado, el objeto lanzado horizontalmente tiene una velocidad inicial positiva.

Como resultado, el objeto lanzado horizontalmente tiene una mayor aceleración debido a su mayor velocidad inicial. Esto significa que se acelera más rápidamente que el objeto caído desde la altura. A medida que el objeto lanzado horizontalmente se acerca al suelo, su velocidad también se acelera a medida que se mueve hacia el suelo.

Finalmente, al objeto lanzado horizontalmente, su aceleración se reduce a la velocidad final, que es igual a la velocidad inicial (la velocidad en que fue lanzado). Por lo tanto, el objeto lanzado horizontalmente se toca con el objeto caído desde la altura al mismo tiempo, porque ambos se aceleran hacia el suelo a velocidades diferentes, pero también a velocidades constantes en el tiempo.

Espero que esto te haya ayudado a entender mejor el concepto de aceleración y cómo afecta el comportamiento de los objetos en caída y lanzamiento.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

Ninguno

### Ronda 1

Excelente observación, revisor. Aquí te presento la versión corregida:

Explica: por qué un objeto en caída libre y otro lanzado horizontalmente desde la misma altura tocan el suelo al mismo tiempo

Estudiante: vamos a analizar el caso de cómo un objeto cae desde una altura y un objeto lanzado horizontalmente desde la misma altura se tocan.

Quiero explicarte un concepto llamado "aceleración" (en inglés, acceleration). Es una fuerza que actúa en cualquier objeto que se mueve de una velocidad inicial a una velocidad más alta. En el caso de los dos objetos, la aceleración es la misma en ambos casos.

Imagina que el objeto caído desde la altura se mueve en una línea recta, con una velocidad inicial. A medida que cae, la aceleración de su proyectil es la gravedad, que es la fuerza que actúa a través de la distancia (en este caso, la altura). La aceleración es constante, es decir, no cambia con el tiempo, y es igual a la razón entre la altura de la caída y el tiempo que tarda en caer.

Por otro lado, el objeto lanzado horizontalmente desde la misma altura también se mueve en una línea recta, pero su velocidad aumenta constantemente debido a la fuerza del impulso. La aceleración en este caso es la velocidad inicial dividida por el tiempo que tarda en caer, o equivalente, la velocidad inicial dividida por la altura.

Ahora, la clave está en la velocidad inicial. El objeto caído desde la altura tiene una velocidad inicial negativa, es decir, se mueve hacia atrás. Por otro lado, el objeto lanzado horizontalmente tiene una velocidad inicial positiva.

Como resultado, el objeto lanzado horizontalmente tiene una mayor aceleración debido a su mayor velocidad inicial. Esto significa que se acelera más rápidamente que el objeto caído desde la altura. A medida que el objeto lanzado horizontalmente se acerca al suelo, su velocidad también se acelera a medida que se mueve hacia el suelo.

Finalmente, al objeto lanzado horizontalmente, su aceleración se reduce a la velocidad final, que es igual a la velocidad inicial (la velocidad en que fue lanzado). Por lo tanto, el objeto lanzado horizontalmente se toca con el objeto caído desde la altura al mismo tiempo, porque ambos se aceleran hacia el suelo a velocidades diferentes, pero también a velocidades constantes en el tiempo.

Espero que esta versión corregida sea más clara y precisa.

... (14 outputs left out)

Tienes razón, la forma en que se expresa la ecuación de velocidad puede ser más clara y concisa. Aquí te dejo la versión corregida completa:

La velocidad es una medida de la cantidad de movimiento que un objeto ha realizado en un intervalo de tiempo determinado. La ecuación de velocidad es una relación entre la distancia recorrida y el tiempo que la objeto ha pasado. En otras palabras, la velocidad es la distancia recorrida dividida por el tiempo. En otras palabras, la velocidad no necesariamente se divide por el tiempo, pero en la ecuación se debe indicar la relación entre la distancia y el tiempo.

Gracias por la corrección, me ayudará a mejorar mi explicación.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA

Comentario: La forma en que se expresa la ecuación de velocidad debe ser más clara, ya que puede ser confusa para algunos estudiantes. La expresión "distancia recorrida dividida por el tiempo" puede ser un poco ambigua. También podría haber mencionado que la velocidad se refiere a la cantidad de movimiento realizado en un intervalo de tiempo determinado, y que la ecuación es una relación entre esta cantidad y el tiempo.

### Ronda 3

¡Entendido! Gracias por señalar la confusión. Aquí te dejo la versión corregida:

La velocidad es una medida de la cantidad de movimiento que un objeto ha realizado en un intervalo de tiempo determinado. La ecuación de velocidad es una relación entre la distancia recorrida y el tiempo que la objeto ha pasado. En otras palabras, la velocidad se refiere a la cantidad de movimiento que un objeto realiza en un intervalo de tiempo determinado, y se mide en metros por segundo (m/s). La ecuación de velocidad es la distancia recorrida dividida por el tiempo. En otras palabras, la velocidad no necesariamente se divide por el tiempo, pero en la ecuación se debe indicar la relación entre la distancia y el tiempo.

Gracias por tu feedback, te ayudará a mejorar la claridad y precisión de mi explicación.

## 5. Modelo pequeno vs modelo mas grande

`llama3.2:1b` es rapido pero poco confiable, y eso se nota **en los dos roles**: el explicador puede cometer errores, y el revisor puede no detectarlos (o rechazar explicaciones que si estaban bien). Un modelo mas grande, como `qwen3:1.7b`, suele ser mas consistente en ambos roles -- sin tocar el resto del codigo, solo cambia el parametro `model`.

In [5]:
model_grande = "qwen3:1.7b"
trace = run_reflection(concepto, model=model_grande, max_rounds=3)
show_trace(concepto, trace)

## Concepto: por que el periodo de un pendulo simple no depende de la masa

### Ronda 0

El periodo de un péndulo simple no depende de la masa porque la aceleración que experimenta el péndulo está determinada por la gravedad y la longitud del hilo, no por la masa. Según la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $, la masa del cuerpo del péndulo se cancela en el cálculo de la aceleración. Esto se debe a que la fuerza gravitacional actúa sobre la masa, pero la aceleración resultante es proporcional a la gravedad dividida por la longitud del péndulo. Por lo tanto, cualquier masa en el péndulo produce la misma aceleración y, por ende, el mismo periodo, independientemente de su masa.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA:  
COMENTARIOS: La explicación es correcta en general, pero podría mejorar resaltando que la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $ se deriva de la aceleración tangencial $ a = g \sin\theta $, y que la masa no aparece en la derivación, aunque su efecto se anula debido a la invariancia de la gravedad en todos los materiales.

### Ronda 1

La explicación es correcta en general, pero se puede mejorar resaltando que la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $ se deriva de la relación entre la aceleración tangencial $ a = g \sin\theta $ y la longitud del péndulo, sin involucrar la masa. Aunque la masa del cuerpo del péndulo afecta la magnitud de la fuerza gravitacional (ya que $ F = mg $), la aceleración resultante $ a = \frac{g \sin\theta}{m} $ se anula debido a que la masa cancela en el cálculo. Esto se debe a que la aceleración tangencial depende solo de la gravedad y la longitud del hilo, no de la masa del cuerpo. Por lo tanto, el periodo del péndulo es independiente de la masa, ya que la invariancia de la gravedad asegura que cualquier material, con masa diferente, produzca la misma aceleración y, por ende, el mismo periodo. La derivación no incluye la masa, pero su efecto se anula en el cálculo físico, confirmando la independencia del periodo de la masa.

... (2 outputs left out)

La explicación es correcta, pero se puede mejorar resaltando que la masa no aparece en la derivación de la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $, ya que la aceleración tangencial $ a = g \sin\theta $ es independiente de la masa. Aunque la masa del cuerpo del péndulo afecta la magnitud de la fuerza gravitacional $ F = mg $, la aceleración resultante $ a = \frac{g \sin\theta}{m} $ se anula en el cálculo físico, ya que la masa cancela en el denominador. Esto se debe a que la aceleración tangencial depende solo de la gravedad y la longitud del hilo, no de la masa del cuerpo. Por lo tanto, el periodo del péndulo es independiente de la masa, ya que la invariancia de la aceleración en función de la gravedad asegura que cualquier material, con masa diferente, produzca la misma aceleración y, por ende, el mismo periodo. La derivación no incluye la masa, pero su efecto se anula en el cálculo físico, confirmando la independencia del periodo de la masa.

**Revision del critico -- Rechazada, se corrige en la siguiente ronda**

APROBADA  
COMENTARIOS: La explicación es correcta, pero se podría mejorar resaltando que la masa no aparece en la derivación de la fórmula, ya que la aceleración tangencial $ a = \frac{g \sin\theta}{m} $ se anula en el cálculo físico, confirmando la independencia del periodo de la masa. La frase "se anula en el cálculo físico" es un poco vaga, pero el mensaje general es claro.

### Ronda 3

La explicación es correcta, pero se puede mejorar resaltando que la masa no aparece en la derivación de la fórmula $ T = 2\pi \sqrt{\frac{L}{g}} $, ya que la aceleración tangencial $ a = \frac{g \sin\theta}{m} $ se cancela en el cálculo físico, confirmando que el periodo no depende de la masa. La masa no se incluye en la derivación, pero su efecto se anula en el cálculo, lo cual es clave para entender por qué el periodo del péndulo es independiente de la masa del cuerpo. Este resultado se debe a que la aceleración resultante depende solo de la gravedad y la longitud del hilo, no de la masa del cuerpo. La frase "se cancela en el cálculo físico" se puede reemplazar con "se anula en el cálculo" para eliminar ambigüedad y mejorar la claridad.


La idea central -- **un LLM que critica y corrige su propia salida antes de entregarla** -- se traslada a otros flujos de un curso o laboratorio:

- **Retroalimentacion de tareas**: el agente redacta un comentario sobre la respuesta de un estudiante, y un segundo paso revisa que la retroalimentacion sea justa y este bien fundamentada antes de mostrarsela.
- **Otros conceptos**: solo cambia el texto que le pasas a `run_reflection`, no hace falta tocar el codigo.
- **Otro criterio de revision**: cambia `REVISOR_PROMPT` en `reflection_agent.py` para que el revisor chequee otra cosa (ej. que la explicacion use el vocabulario correcto para el nivel del curso, o que incluya una analogia).
- **Combinarlo con herramientas**: se puede mezclar con el patron de `Tools` -- por ejemplo, un agente que ajusta una curva y *despues* reflexiona sobre si el modelo elegido tiene sentido fisico.